In [1]:
# ---------------------------
# CONFIG (edit these values)
# ---------------------------
import os
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_curve, confusion_matrix

# =========================================
# GLOBAL SETTINGS - Edit for new analyses
# =========================================
seasons = ["PREMONSOON","MONSOON"]
use_water_level_features=True
years_to_process =  [2017,2022,2024]  # List of years to process (e.g., [2017, 2018, 2019, 2020, 2021, 2022])
output_group = "uh_based"

try:
    watershed_area_km2 = float(input("Enter watershed area (km^2): ").strip())
    if watershed_area_km2 <= 0:
        raise ValueError("Watershed area must be positive.")
except Exception:
    watershed_area_km2 = 1.0
    print("Invalid watershed area input. Using default: 1.0 km^2")

try:
    tp = float(input("Enter Tp (hours): " ).strip())
    if tp <= 0:
        raise ValueError("Tp must be positive.")
except Exception:
    tp = 17.0
    print("Invalid Tp input. Using default: 17.0 hours")

try:
    tb = float(input("Enter Tb (hours): " ).strip())
    if tb <= 0:
        raise ValueError("Tb must be positive.")
except Exception:
    tb = 45.0
    print("Invalid Tb input. Using default: 45.0 hours")

# Base configuration function (will be called for each year)
def create_config(
    year,
    season,
    watershed_area_km2,
):
    return {
        "forecast_excel": Path("..") / "LSTM" / "ECMWF_HRES_forecast_excel" / f"{year}_january_july.xlsx",
        "obs_excel": Path(f"D:/Masters/data/predicting_flashflood_events_codes/2017-24obsexcel.xlsx"),
        "predictions_dir": Path(f"predictions_hres_tp{tp}_tb{tb}") / output_group / f"{season}",
        "plots_dir": Path(f"plots_HRES_tp{tp}_tb{tb}") / output_group / f"{season}",
        "watershed_area_km2": watershed_area_km2,
    }

# Create initial CONFIG
CONFIG = create_config(
    years_to_process[0],
    seasons[0],  # Use the first season for initial config; will loop through seasons later
    watershed_area_km2
)

# Create folders
for p in (CONFIG["predictions_dir"], CONFIG["plots_dir"]):
    os.makedirs(p, exist_ok=True)



print("=" * 70)
print("CONFIGURATION LOADED")
print("=" * 70)
print(f"Years to process: {years_to_process}")

print(f"Output group: {output_group}")
print(f"Watershed area: {watershed_area_km2} km^2")
print(f"Tp: {tp} hours | Tb: {tb} hours")

CONFIGURATION LOADED
Years to process: [2017, 2022, 2024]
Output group: uh_based
Watershed area: 785.0 km^2
Tp: 25.0 hours | Tb: 110.0 hours


In [2]:
# ---------------------------
# Helpers (UH, IO)
# ---------------------------
import numpy as np
import pandas as pd


class UHFunction:
    def __init__(self, watershed_area_km2):
        self.area = float(watershed_area_km2)
        self.Tp = float(globals().get("tp", 17.0))  # hours
        self.Tb = float(globals().get("tb", 45.0))  # hours
        self.phi_monsoon = 0.5  # mm/hr (Jun-Sep)
        self.phi_premonsoon = 1.5  # mm/hr (Mar-May)
        self.phi_postmonsoon = 1.5  # mm/hr (Oct-Nov)
        self.phi_winter = 1.5  # mm/hr (Dec-Feb)

    def get_seasonal_phi(self, month):
        if month in [6, 7, 8, 9]:
            return self.phi_monsoon
        if month in [3, 4, 5]:
            return self.phi_premonsoon
        if month in [10, 11]:
            return self.phi_postmonsoon
        return self.phi_winter

    def triangular_uh(self, t):
        Qp = 2.0 / self.Tb
        h = np.zeros_like(t, dtype=float)
        rising = (t >= 0) & (t <= self.Tp)
        recession = (t > self.Tp) & (t <= self.Tb)
        h[rising] = (Qp / self.Tp) * t[rising]
        h[recession] = Qp - (Qp / (self.Tb - self.Tp)) * (t[recession] - self.Tp)
        beyond_base = t >self.Tb

        h[beyond_base] = 0.0

        # Safety clip
        h = np.clip(h, 0, None)
        return h

    def calculate_effective_rainfall(self, df, precip_col="precip"):
        out = df.copy()
        out["phi_index"] = out.index.month.map(self.get_seasonal_phi)
        out["P_effective"] = (out[precip_col] - out["phi_index"]).clip(lower=0)
        return out

    def simulate_discharge(self, df, precip_col="precip"):
        out = self.calculate_effective_rainfall(df, precip_col=precip_col)
        max_time = int(self.Tb * 3)
        t_uh = np.arange(0, max_time, 1.0)
        uh = self.triangular_uh(t_uh)
        uh_m3s_mm = uh * self.area * 1000.0 / 3600.0
        p_eff = out["P_effective"].to_numpy(dtype=float)
        q_direct = np.convolve(p_eff, uh_m3s_mm, mode="full")[: len(p_eff)]
        out["Q_direct_simulated"] = q_direct
        return out


def get_end_day(month: int) -> int:
    return 31 if month in [1, 3, 5, 7, 8, 10, 12] else (30 if month in [4, 6, 9, 11] else 29)


def get_season_month_window(season_name: str):
    season_norm = str(season_name).strip().upper()
    if season_norm == "PREMONSOON":
        return 3, 5
    if season_norm == "MONSOON":
        return 6, 7
    return None, None


def get_obs_columns(df_obs: pd.DataFrame):
    wl_col = "WL (mMSL)" if "WL (mMSL)" in df_obs.columns else None

    q_candidates = [
        "Q",
        "Q_obs",
        "Q_observed",
        "Q (m3/s)",
        "Discharge",
        "discharge",
    ]
    q_col = next((c for c in q_candidates if c in df_obs.columns), None)

    if q_col is None:
        for c in df_obs.columns:
            c_norm = str(c).strip().lower()
            if c_norm == "q" or c_norm.startswith("q_") or "discharge" in c_norm:
                q_col = c
                break

    return wl_col, q_col


# Safe default to avoid NameError in later cells that call: season=season
if "season" not in globals():
    _season_from_config = None
    if "CONFIG" in globals() and isinstance(CONFIG, dict):
        pred_dir = CONFIG.get("predictions_dir", None)
        if pred_dir is not None:
            try:
                _season_from_config = str(pred_dir.name).upper()
            except Exception:
                _season_from_config = None
    season = _season_from_config


def build_forecast_matrix(
    forecast_file,
    obs_excel,
    year,
    end_month,
    watershed_area_km2,
    season=None,
    series_type="x",
    obs_precip_col="corr_precip",
    history_hours=240,
    forecast_hours=240,
):
    """Build lead-time matrix. For discharge, use obs precip (-240h) + forecast precip (+240h) per init time."""
    df_fcst = pd.read_excel(forecast_file)
    if "date" in df_fcst.columns:
        df_fcst = df_fcst.drop(columns=["date"])

    df_obs_precip = pd.read_excel(obs_excel)
    if "Date Time" not in df_obs_precip.columns:
        raise KeyError("obs_excel must contain 'Date Time' column")
    if obs_precip_col not in df_obs_precip.columns:
        raise KeyError(f"obs_excel must contain '{obs_precip_col}' column")

    df_obs_precip["Date Time"] = pd.to_datetime(df_obs_precip["Date Time"])
    df_obs_precip = df_obs_precip.set_index("Date Time").sort_index()

    # Use season to define month window (so `season` is actually used)
    season_start, season_end = get_season_month_window(season) if season is not None else (None, None)
    if end_month is not None:
        if season_end is None:
            season_end = int(end_month)
        else:
            season_end = min(int(season_end), int(end_month))

    uh_model = UHFunction(watershed_area_km2=watershed_area_km2)
    hours = np.arange(0, forecast_hours + 1, 6)
    df_pred = pd.DataFrame(index=np.arange(0, forecast_hours + 1, 1))

    for col_name in df_fcst.columns:
        try:
            init_time = pd.to_datetime(col_name)
        except Exception:
            continue

        # Filter by target year and month window
        if year is not None and init_time.year != int(year):
            continue
        if season_start is not None and season_end is not None:
            if not (int(season_start) <= init_time.month <= int(season_end)):
                continue
        elif end_month is not None and init_time.month > int(end_month):
            continue

        index_s = [init_time + pd.Timedelta(hours=int(h)) for h in hours]
        df_block = pd.DataFrame({"value": df_fcst[col_name].values}, index=index_s)
        df_block["increment"] = df_block["value"].diff().fillna(df_block["value"])
        df_hourly_fcst = df_block.resample("h").ffill()
        df_hourly_fcst["precip"] = df_hourly_fcst["increment"] / 6

        future_idx = pd.date_range(
            start=init_time,
            end=init_time + pd.Timedelta(hours=forecast_hours),
            freq="h",
        )
        future_precip = df_hourly_fcst["precip"].reindex(future_idx).ffill().fillna(0.0)

        if series_type == "precip":
            series = future_precip
        else:
            past_idx = pd.date_range(
                start=init_time - pd.Timedelta(hours=history_hours),
                end=init_time - pd.Timedelta(hours=1),
                freq="h",
            )
            past_precip = (
                df_obs_precip[obs_precip_col]
                .reindex(past_idx)
                .interpolate(method="time", limit_direction="both")
                .fillna(0.0)
            )

            combined_precip = pd.concat([past_precip, future_precip])
            combined_df = pd.DataFrame({"precip": combined_precip})
            discharge_df = uh_model.simulate_discharge(combined_df, precip_col="precip")

            

            series = discharge_df.loc[future_idx, "Q_direct_simulated"]

        df_pred[col_name] = series.to_numpy(dtype=float)

    df_pred_T = df_pred.T
    df_pred_T.index = pd.to_datetime(df_pred_T.index, errors="coerce")
    df_pred_T = df_pred_T[~df_pred_T.index.isna()]
    return df_pred_T

In [3]:
for year in years_to_process:
    for season in seasons:
        print(f"\n{'='*70}")
        print(f"Processing Year: {year}")
        print(f"Season: {season}")
        print(f"{'='*70}")

        CONFIG = create_config(
            year,
            season,
            watershed_area_km2,
        )
        os.makedirs(CONFIG["predictions_dir"], exist_ok=True)
        os.makedirs(CONFIG["plots_dir"], exist_ok=True)

        forecast_file = CONFIG["forecast_excel"]
        obs_file = CONFIG["obs_excel"]

        if not forecast_file.exists():
            print(f"Forecast file not found: {forecast_file}")
            continue
        if not obs_file.exists():
            print(f"Observation file not found: {obs_file}")
            continue

        display_start_month, display_end_month = get_season_month_window(season)
        if display_start_month is None or display_end_month is None:
            print(f"Unsupported season '{season}'. Skipping year {year}.")
            continue

        end_month = display_end_month
        display_end_day = get_end_day(display_end_month)

        # Import plotly EARLY, before any plotting code
        try:
            import plotly.graph_objects as go
            from plotly.subplots import make_subplots
        except ImportError:
            print("Plotly is not installed in this kernel. Install with: pip install plotly")
            continue

        # Load observed data early (needed for plotting)
        df_obs = pd.read_excel(obs_file)
        df_obs["Date Time"] = pd.to_datetime(df_obs["Date Time"])
        df_obs = df_obs.set_index("Date Time").sort_index()
        wl_col, q_col = get_obs_columns(df_obs)

        # Define plot window early
        plot_start = pd.Timestamp(year=year, month=display_start_month, day=1, hour=0, minute=0, second=0)
        plot_end = pd.Timestamp(year=year, month=display_end_month, day=display_end_day, hour=23, minute=59, second=59)

        print(f"Loading forecasts from: {forecast_file}")
        df_pred_T = build_forecast_matrix(
            forecast_file=forecast_file,
            obs_excel=obs_file,
            year=year,
            end_month=end_month,
            watershed_area_km2=CONFIG["watershed_area_km2"],
            season=season,
            series_type="x",
            obs_precip_col="corr_precip",
            history_hours=240,
            forecast_hours=240,
        )

        df_pred_excel_path = CONFIG["predictions_dir"] / f"df_pred_T_{year}_{season}.xlsx"
        df_pred_T.to_excel(df_pred_excel_path, index=True)
        print(f"Saved df_pred_T to: {df_pred_excel_path}")

        # Extract selected lead times early
        selected_leads = []
        for c in df_pred_T.columns:
            try:
                lead = int(c)
                if lead > 0 and lead % 24 == 0:
                    selected_leads.append(lead)
            except Exception:
                continue
        selected_leads = sorted(set(selected_leads))

        if len(selected_leads) == 0:
            print(f"No 24-hour interval lead times found for year {year}.")
            continue

        # Build forecast precipitation matrix (hourly mm values per lead hour)
        print(f"Loading forecast precipitation from: {forecast_file}")
        df_pred_T_precip = build_forecast_matrix(
            forecast_file=forecast_file,
            obs_excel=obs_file,
            year=year,
            end_month=end_month,
            watershed_area_km2=CONFIG["watershed_area_km2"],
            season=season,
            series_type="precip",
            obs_precip_col="corr_precip",
            history_hours=240,
            forecast_hours=240,
        )

        # Now build and plot discharge figure
        fig = make_subplots(specs=[[{"secondary_y": True}]])
        plotted_ts = []

        for lead in selected_leads:
            col = lead if lead in df_pred_T.columns else str(lead)
            if col not in df_pred_T.columns:
                continue

            ts = df_pred_T.index + pd.Timedelta(hours=lead)
            sim = pd.to_numeric(df_pred_T[col], errors="coerce").values
            mask = ~np.isnan(sim)
            if mask.sum() == 0:
                continue

            ts_m = ts[mask]
            sim_m = sim[mask]
            in_window = (ts_m >= plot_start) & (ts_m <= plot_end)
            if in_window.sum() == 0:
                continue

            ts_m = ts_m[in_window]
            sim_m = sim_m[in_window]
            plotted_ts.append(ts_m)
            fig.add_trace(
                go.Scatter(
                    x=ts_m,
                    y=sim_m,
                    mode="lines",
                    name=f"Q_simulated (lead {lead}h)",
                    line=dict(width=1),
                ),
                secondary_y=False,
            )

        if len(plotted_ts) > 0:
            ts_union = pd.DatetimeIndex(np.unique(np.concatenate([ts.values for ts in plotted_ts])))

            if wl_col is not None:
                wl_series = df_obs.reindex(ts_union)[wl_col]
                wl_valid = ~wl_series.isna()
                if wl_valid.sum() > 0:
                    fig.add_trace(
                        go.Scatter(
                            x=ts_union[wl_valid],
                            y=wl_series[wl_valid],
                            mode="lines",
                            name=wl_col,
                            line=dict(width=2),
                        ),
                        secondary_y=True,
                    )

            if q_col is not None:
                q_series = df_obs.reindex(ts_union)[q_col]
                q_valid = ~q_series.isna()
                if q_valid.sum() > 0:
                    fig.add_trace(
                        go.Scatter(
                            x=ts_union[q_valid],
                            y=q_series[q_valid],
                            mode="lines",
                            name=q_col,
                            line=dict(width=2, dash="dash"),
                        ),
                        secondary_y=False,
                    )

        fig.update_xaxes(
            title_text="Time",
            range=[plot_start, plot_end],
            showline=True,
            linecolor="black",
            linewidth=2,
            mirror=True,
            ticks="outside",
            tickcolor="black",
            showgrid=True,
            gridcolor="rgba(120,120,120,0.30)",
            gridwidth=1,
        )
        fig.update_yaxes(
            title_text="Q (simulated/observed)",
            secondary_y=False,
            showline=True,
            linecolor="black",
            linewidth=2,
            mirror=True,
            ticks="outside",
            tickcolor="black",
            showgrid=True,
            gridcolor="rgba(120,120,120,0.30)",
            gridwidth=1,
        )
        fig.update_yaxes(
            title_text=wl_col if wl_col is not None else "WL",
            secondary_y=True,
            showline=True,
            linecolor="black",
            linewidth=2,
            mirror=True,
            ticks="outside",
            tickcolor="black",
            showgrid=False,
        )
        fig.update_layout(
            hovermode="x unified",
            template="simple_white",
            legend=dict(
                orientation="h",
                yanchor="bottom",
                y=1.02,
                xanchor="left",
                x=0,
                font=dict(size=13),
            ),
        )

        html_plot_path = CONFIG["plots_dir"] / f"interactive_leads_{year}_{season}.html"
        fig.write_html(str(html_plot_path), include_plotlyjs="cdn")
        print(f"Saved interactive plot to: {html_plot_path}")

        # Save as high-resolution SVG
        try:
            svg_plot_path = CONFIG["plots_dir"] / f"interactive_leads_{year}_{season}.svg"
            fig.write_image(str(svg_plot_path), width=1400, height=700)
            print(f"Saved high-resolution SVG plot to: {svg_plot_path}")
        except Exception as e:
            print(f"Could not save SVG (kaleido may need to be installed): {e}")

        if q_col is None:
            print("Observed Q column not found in obs_excel. Plotted WL only.")

        fig.show()

        # PRECIP VS wl/q plot with bars for 24h-accumulated forecast precip and lines for wl/q
        fig = make_subplots(specs=[[{"secondary_y": True}]])
        plotted_ts = []
        max_precip = 0.0

        for lead in selected_leads:
            prev_lead = lead - 24
            window_cols = [
                c for c in df_pred_T_precip.columns
                if isinstance(c, (int, np.integer)) and prev_lead < c <= lead
            ]
            if len(window_cols) == 0:
                continue

            precip_24h = df_pred_T_precip[window_cols].sum(axis=1)
            ts = df_pred_T_precip.index + pd.Timedelta(hours=lead)
            sim = pd.to_numeric(precip_24h, errors="coerce").values
            mask = ~np.isnan(sim)
            if mask.sum() == 0:
                continue

            ts_m = ts[mask]
            sim_m = sim[mask]
            in_window = (ts_m >= plot_start) & (ts_m <= plot_end)
            if in_window.sum() == 0:
                continue

            ts_m = ts_m[in_window]
            sim_m = sim_m[in_window]
            plotted_ts.append(ts_m)
            max_precip = max(max_precip, float(np.nanmax(sim_m)))

            fig.add_trace(
                go.Bar(
                    x=ts_m,
                    y=-sim_m,
                    name=f"precip (lead {lead}h)",
                    opacity=0.45,
                ),
                secondary_y=False,
            )

        if len(plotted_ts) > 0:
            ts_union = pd.DatetimeIndex(np.unique(np.concatenate([ts.values for ts in plotted_ts])))

            if wl_col is not None:
                wl_series = df_obs.reindex(ts_union)[wl_col]
                wl_valid = ~wl_series.isna()
                if wl_valid.sum() > 0:
                    fig.add_trace(
                        go.Scatter(
                            x=ts_union[wl_valid],
                            y=wl_series[wl_valid],
                            mode="lines",
                            name=wl_col,
                            line=dict(width=2),
                        ),
                        secondary_y=True,
                    )

            if q_col is not None:
                q_series = df_obs.reindex(ts_union)[q_col]
                q_valid = ~q_series.isna()
                if q_valid.sum() > 0:
                    fig.add_trace(
                        go.Scatter(
                            x=ts_union[q_valid],
                            y=q_series[q_valid],
                            mode="lines",
                            name=q_col,
                            line=dict(width=2, dash="dash"),
                        ),
                        secondary_y=True,
                    )

        fig.update_xaxes(
            title_text="Time",
            range=[plot_start, plot_end],
            showline=True,
            linecolor="black",
            linewidth=2,
            mirror=True,
            ticks="outside",
            tickcolor="black",
            showgrid=True,
            gridcolor="rgba(120,120,120,0.30)",
            gridwidth=1,
        )
        fig.update_yaxes(
            title_text="24h accumulated precipitation forecast (mm)",
            secondary_y=False,
            showline=True,
            linecolor="black",
            linewidth=2,
            mirror=True,
            ticks="outside",
            tickcolor="black",
            showgrid=True,
            gridcolor="rgba(120,120,120,0.30)",
            gridwidth=1,
        )
        fig.update_yaxes(
            title_text="Observed WL/Q",
            secondary_y=True,
            showline=True,
            linecolor="black",
            linewidth=2,
            mirror=True,
            ticks="outside",
            tickcolor="black",
            showgrid=False,
        )

        if max_precip > 0:
            tick_vals = np.linspace(-max_precip, 0, 6)
            tick_text = [f"{abs(v):.1f}" for v in tick_vals]
            fig.update_yaxes(
                tickmode="array",
                tickvals=tick_vals,
                ticktext=tick_text,
                secondary_y=False,
            )

        fig.update_layout(
            title=f"Year {year} | 24h Accumulated Forecast Precipitation (mm) with Observed WL and Q - Interactive",
            hovermode="x unified",
            template="simple_white",
            barmode="overlay",
            legend=dict(
                orientation="h",
                yanchor="bottom",
                y=1.02,
                xanchor="left",
                x=0,
                font=dict(size=16),
            ),
        )

        html_plot_path = CONFIG["plots_dir"] / f"interactive_precipleads_inverted_{year}_{season}.html"
        fig.write_html(str(html_plot_path), include_plotlyjs="cdn")
        print(f"Saved interactive inverted precip plot to: {html_plot_path}")

        # Save as high-resolution SVG
        try:
            svg_plot_path = CONFIG["plots_dir"] / f"interactive_precipleads_inverted_{year}_{season}.svg"
            fig.write_image(str(svg_plot_path), width=1400, height=700)
            print(f"Saved high-resolution SVG plot to: {svg_plot_path}")
        except Exception as e:
            print(f"Could not save SVG (kaleido may need to be installed): {e}")

        if q_col is None:
            print("Observed Q column not found in obs_excel. Plotted WL only.")

        fig.show()


Processing Year: 2017
Season: PREMONSOON
Loading forecasts from: ..\LSTM\ECMWF_HRES_forecast_excel\2017_january_july.xlsx
Saved df_pred_T to: predictions_hres_tp25.0_tb110.0\uh_based\PREMONSOON\df_pred_T_2017_PREMONSOON.xlsx
Loading forecast precipitation from: ..\LSTM\ECMWF_HRES_forecast_excel\2017_january_july.xlsx
Saved interactive plot to: plots_HRES_tp25.0_tb110.0\uh_based\PREMONSOON\interactive_leads_2017_PREMONSOON.html
Could not save SVG (kaleido may need to be installed): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



Saved interactive inverted precip plot to: plots_HRES_tp25.0_tb110.0\uh_based\PREMONSOON\interactive_precipleads_inverted_2017_PREMONSOON.html
Could not save SVG (kaleido may need to be installed): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido




Processing Year: 2017
Season: MONSOON
Loading forecasts from: ..\LSTM\ECMWF_HRES_forecast_excel\2017_january_july.xlsx
Saved df_pred_T to: predictions_hres_tp25.0_tb110.0\uh_based\MONSOON\df_pred_T_2017_MONSOON.xlsx
Loading forecast precipitation from: ..\LSTM\ECMWF_HRES_forecast_excel\2017_january_july.xlsx
Saved interactive plot to: plots_HRES_tp25.0_tb110.0\uh_based\MONSOON\interactive_leads_2017_MONSOON.html
Could not save SVG (kaleido may need to be installed): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



Saved interactive inverted precip plot to: plots_HRES_tp25.0_tb110.0\uh_based\MONSOON\interactive_precipleads_inverted_2017_MONSOON.html
Could not save SVG (kaleido may need to be installed): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido




Processing Year: 2022
Season: PREMONSOON
Loading forecasts from: ..\LSTM\ECMWF_HRES_forecast_excel\2022_january_july.xlsx
Saved df_pred_T to: predictions_hres_tp25.0_tb110.0\uh_based\PREMONSOON\df_pred_T_2022_PREMONSOON.xlsx
Loading forecast precipitation from: ..\LSTM\ECMWF_HRES_forecast_excel\2022_january_july.xlsx
Saved interactive plot to: plots_HRES_tp25.0_tb110.0\uh_based\PREMONSOON\interactive_leads_2022_PREMONSOON.html
Could not save SVG (kaleido may need to be installed): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



...

Saved interactive inverted precip plot to: plots_HRES_tp25.0_tb110.0\uh_based\MONSOON\interactive_precipleads_inverted_2022_MONSOON.html
Could not save SVG (kaleido may need to be installed): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido




Processing Year: 2024
Season: PREMONSOON
Loading forecasts from: ..\LSTM\ECMWF_HRES_forecast_excel\2024_january_july.xlsx
Saved df_pred_T to: predictions_hres_tp25.0_tb110.0\uh_based\PREMONSOON\df_pred_T_2024_PREMONSOON.xlsx
Loading forecast precipitation from: ..\LSTM\ECMWF_HRES_forecast_excel\2024_january_july.xlsx
Saved interactive plot to: plots_HRES_tp25.0_tb110.0\uh_based\PREMONSOON\interactive_leads_2024_PREMONSOON.html
Could not save SVG (kaleido may need to be installed): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



Saved interactive inverted precip plot to: plots_HRES_tp25.0_tb110.0\uh_based\PREMONSOON\interactive_precipleads_inverted_2024_PREMONSOON.html
Could not save SVG (kaleido may need to be installed): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido




Processing Year: 2024
Season: MONSOON
Loading forecasts from: ..\LSTM\ECMWF_HRES_forecast_excel\2024_january_july.xlsx
Saved df_pred_T to: predictions_hres_tp25.0_tb110.0\uh_based\MONSOON\df_pred_T_2024_MONSOON.xlsx
Loading forecast precipitation from: ..\LSTM\ECMWF_HRES_forecast_excel\2024_january_july.xlsx
Saved interactive plot to: plots_HRES_tp25.0_tb110.0\uh_based\MONSOON\interactive_leads_2024_MONSOON.html
Could not save SVG (kaleido may need to be installed): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



Saved interactive inverted precip plot to: plots_HRES_tp25.0_tb110.0\uh_based\MONSOON\interactive_precipleads_inverted_2024_MONSOON.html
Could not save SVG (kaleido may need to be installed): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido

